# Step 1 - Prototype 2

We now have our first working prototype for step 1. In the second (and probably final) one, we mainly want to fix some smaller issues and decide on foundational formulations / implementations that will make extendability much easier later on. These are:
- Think of a good way to encode and implement train stations (probably already along with the corresponding constraints (essentially blocking movement until the leave time)).
- Adjust the optimization goal to minimize lateness (for now only at the destination station). Additionally, maybe make sure that the formulation can be extended in terms of travel and dwell time minimization goals later.

Adding a safety distance now might also be doable, however, this is in a sense then also connected to different train speeds (that might require different safety distances) and this is probably really then a point for step 2...

Small terminology recap (how I use it here):
- Track: What a train drives on
- Block: A section of tracks (1 or more) of fixed length (like if you would have a map on checkered paper)
- Segment: A sequence of (or also only one xD) block(s) with the same capacity

In [16]:
import gurobipy as gp
from gurobipy import GRB

import numpy as np
from enum import Enum

In [17]:
# Initialize the Model
model = gp.Model("Basic_MILP")

### Define the track system and timetable

Here we want to see, if it makes sense to create (separate?) definitions for train stations and block segments that can then hopefully be combined automatically or, if just adding segments of length one with a given 'station capacity' should work just fine (or even better xD).

Given the fact, that we probably only want to have leave and arrival times at stations and not on 'random' points on the tracks, a more native way of distinguishing (separate blocks and train stations) might be beneficial. This could then be handled via an additional boolean array `is_train_station_array` (or something like that xD).

**Another Idea:** `segment_lengths` is a list of lists (and single entries?). The same goes for segment capacities and per construction, there is always a station (with capacities specified in `station_capacities`) between a given connecting segment. This means, that `num_stations` is equal to `len(segment_lengths)`.

To avoid inhomogeneous arrays, we will use a list of dictionaries and a parser function that turns it into proper flat arrays for Gurobi... This way, we should be able to extend the model and add new features using similar helper / parsing functionalities, as well as later in the constraint generation. 

In [18]:
# Might use Enums or Dataclasses later but maybe also not xD
# class TrackType(Enum):
#     station = "station"
#     segment = "segment"

# Copy preset for faster building
# {"type": "station", "capacity": 0}
# {"type": "segment", "length": 0, "capacity": 0}

track_blueprint = [
    {"type": "station", "capacity": 3},
    {"type": "segment", "length": 5, "capacity": 2},
    {"type": "station", "capacity": 3}
]

In [19]:
# Parsing the track blueprint and generating necessary data

block_capacities = []
is_station = []

for item in track_blueprint:
    if item["type"] == "station":
        block_capacities.append(item["capacity"])
        is_station.append(True)
    elif item["type"] == "segment":
        # Int conversion not necessary, but gets rid of IDE highlighting
        for _ in range(int(item["length"])):
            block_capacities.append(item["capacity"])
            is_station.append(False)
    else:
        print("typo? No other types yet...")

In [20]:
# Like with the boolean mask is_station, we will use a mask is_stop for our timetable as well.
# This will set us up nicely when having different train types (e.g. RB / RE and ICE) later on
# and also allows for a more flexible and complete rule set.

# New Idea: To allow for better extendability, we will try out a similar dictionary-based approach
# like we did with the track_blueprint. This will make it much easier to add essentially arbitrary
# amounts of additional information about each stop (e.g. dwell times, passengers, ...) and never
# have to worry about index-mismatches.

# Structure: [train_i_stop_information_dictionary{station_index_i: information{some_key: some_value}}]
# Note: Due to the implementation ('looking-back') leave_times_i has to be >= 1
train_schedules = [
    # Train 0
    {
        # Station x
        0: {"leave": 3},
        2: {"arrival": 16, "leave": 17},
    },
    # Train 1
    {
        # Station x
        1: {"leave": 5},

        # Think what dwell should / might mean in combination with a leave time ('overpower', something else?)
        # Might not be even implemented now but could be used later on...
        2: {"arrival": 15, "leave": 18, "dwell": 2},
    },
]

In [30]:
# Just a cell for dictionary access testing
print(sorted(train_schedules[1].keys())[0])

spawn_time_key = sorted(train_schedules[0].keys())[0]
spawn_time = train_schedules[0][spawn_time_key]["leave"]

print(spawn_time)

1
3


In [21]:
# Number of time steps over which we optimize
time_horizon = 25

# Deriving some additional variables
num_trains = len(train_schedules)
num_blocks = len(block_capacities)
num_stations = sum(is_station)

In [22]:
# Define Decision Variables

# To describe our train movement we need a way to know which train is where at what
# time (-> 3D time-position grid of boolean values)

x = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x")

### Constraints

#### General Constraints

As general constraints, we have the
- uniqueness constraint (no train can be on multiple tracks at once)
- spawning constraint (a train does not 'exist' until it leaves from the first station in its schedule). For future iterations we should also think about if we want to keep the spawning behavior or replace (or even remove?) it...
- capacity constraints (there can't exist more trains on a block (or station) than its capacity)

#### Movement Constraints

For now, we just have the 'at most one block per timestep' constraint. Will likely be changed in a later version though.

In [ ]:
# Uniqueness constraint for trains to only exist once on a block (on only a single track at a given time)
# Additionally, we tackle the spawning (leave times), as before a train does not exist on any block
for i in range(num_trains):

    # Since dictionaries are unordered, use this to get spawn time (first leave time)
    spawn_time_key = sorted(train_schedules[i].keys())[0]
    spawn_time = train_schedules[i][spawn_time_key]["leave"]

    for k in range(time_horizon):

        # Sum over all blocks j to ensure that a train only exists once on this block
        active_tracks = gp.quicksum(x[i, j, k] for j in range(num_blocks))

        # If the train hasn't spawned (or left) yet, it can't exist on the track and we enforce this
        if k < spawn_time:
            model.addConstr(active_tracks == 0, name=f"NotSpawned_Train{i}_Time{k}")
        else:
            # If it already exists, we enforce that the train only exists on one track at any time k
            model.addConstr(active_tracks == 1, name=f"UniquePosition_Train{i}_Time{k}")

In [ ]:
# Capacity constraints (the sum over all tracks j and all trains i <= capacities_i)
for k in range(time_horizon):

    # Here we want to sum over all i AND j and add the constraint that this should be smaller than
    # Is this formulation correct and can this be made more compact (yes: x.sum('*', j, k))?
    for j in range(num_blocks):
        occupied_tracks = gp.quicksum(x[i, j, k] for i in range(num_trains))

        # Add constraint to enforce capacity maximum across block j
        model.addConstr(occupied_tracks <= block_capacities[j], name=f"Capacity_Block{j}_Time{k}")